# CrewAI Demo: Marketing Platform Content Crew

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/knivok/ai-playbook/blob/main/notebooks/01_frameworks/crewai/05_demo_content_crew.ipynb)

---

This is the CrewAI benchmark demo. Given a product niche: research it, write a blog post, optimize for SEO, and prepare social media adaptations.

This maps directly to Knovik's **Marketing Platform** — the same crew structure used here is how the platform's AI content pipeline works in production.

## The Crew

```
Agent 1: Market Research Analyst  — tools: search, keyword_data
  ↓
Agent 2: Content Strategist       — tools: keyword_data, existing_posts
  ↓
Agent 3: SEO Content Writer       — tools: none (writes from context)
  ↓
Agent 4: SEO Specialist           — tools: keyword_data
  ↓
Agent 5: Social Media Specialist  — tools: none
```

## Compare with LangGraph
CrewAI uses ~3x the tokens of an equivalent LangGraph pipeline.
For Marketing Platform where output quality drives conversions, that trade-off is worth it.


In [ ]:
!pip install -q crewai crewai-tools langchain-anthropic python-dotenv rich

In [ ]:
import os
from getpass import getpass

if not os.getenv('ANTHROPIC_API_KEY'):
    os.environ['ANTHROPIC_API_KEY'] = getpass('Anthropic API key: ')
if not os.getenv('SERPER_API_KEY'):
    os.environ['SERPER_API_KEY'] = getpass('Serper API key (serper.dev — free tier): ')

print('✅ Keys set')

In [ ]:
import json
from crewai import Agent, Task, Crew, Process
from crewai import tool
from crewai_tools import SerperDevTool
from langchain_anthropic import ChatAnthropic
from rich.console import Console
from rich.panel import Panel
from rich.markdown import Markdown

console = Console()

# Haiku for most agents (fast + cheap), Sonnet for the writer (quality matters)
llm_fast = ChatAnthropic(model='claude-3-5-haiku-20241022', temperature=0.1)
llm_quality = ChatAnthropic(model='claude-3-5-sonnet-20241022', temperature=0.3)

print('✅ LLMs configured')

In [ ]:
# ── Custom Tools ──────────────────────────────────────────────────────────

search_tool = SerperDevTool()

@tool('Get keyword SEO data')
def keyword_data_tool(keyword: str) -> str:
    """
    Returns SEO metrics: monthly search volume, difficulty (0-100), and CPC.
    Use for content planning and on-page SEO optimization.
    Args:
        keyword: The keyword or phrase to analyze
    """
    keyword_db = {
        'zigbee hub': {'volume': 18000, 'difficulty': 38, 'cpc': 1.45},
        'zigbee hub homekit': {'volume': 5400, 'difficulty': 29, 'cpc': 1.80},
        'best zigbee hub 2026': {'volume': 8100, 'difficulty': 45, 'cpc': 1.60},
        'homekit zigbee': {'volume': 9900, 'difficulty': 35, 'cpc': 1.75},
        'aqara hub': {'volume': 6600, 'difficulty': 32, 'cpc': 0.95},
        'matter protocol hub': {'volume': 4400, 'difficulty': 28, 'cpc': 2.10},
    }
    data = keyword_db.get(keyword.lower().strip(), {'volume': 2000, 'difficulty': 40, 'cpc': 1.20})
    return json.dumps({
        'keyword': keyword,
        'monthly_searches': data['volume'],
        'difficulty': f"{data['difficulty']}/100 ({'Easy' if data['difficulty'] < 35 else 'Medium' if data['difficulty'] < 55 else 'Hard'})",
        'cpc_usd': data['cpc'],
        'recommendation': 'Target' if data['difficulty'] < 45 else 'Competitive — strong content needed'
    }, indent=2)


@tool('Check existing Knovik blog posts')
def existing_posts_tool(topic: str) -> str:
    """
    Checks for existing Knovik blog posts on a topic to avoid duplicate content
    and identify internal linking opportunities.
    Args:
        topic: Topic to search for (e.g. 'zigbee', 'homekit', 'matter')
    """
    existing = {
        'zigbee': [
            'What is Zigbee? A Beginners Guide (addtohomekit.com/what-is-zigbee)',
            'Top 10 Zigbee Devices for Smart Homes (athbridge.com/blog/top-zigbee-devices)'
        ],
        'homekit': [
            'Apple HomeKit Setup Guide 2026 (addtohomekit.com/setup-guide)',
        ],
        'matter': [
            'Matter Protocol Explained (matterhubs.com/what-is-matter)'
        ]
    }
    found = existing.get(topic.lower().strip(), [])
    if found:
        return f"Existing posts on '{topic}':\n" + '\n'.join(f'- {p}' for p in found) + '\n\nUse for internal linking. Do not duplicate these angles.'
    return f"No existing posts on '{topic}'. Fresh content opportunity."


print('✅ Tools ready')

In [ ]:
# ── Agents ────────────────────────────────────────────────────────────────

market_researcher = Agent(
    role='Senior Market Research Analyst — Consumer Electronics & Smart Home',
    goal='Deliver accurate, quantified market research with real data: market size, growth rate, named competitors, consumer pain points.',
    backstory='"""10 years covering smart home and IoT. You advise hardware startups and e-commerce businesses on product positioning.
You are rigorous about sources and always distinguish facts from estimates."""',
    tools=[search_tool, keyword_data_tool],
    llm=llm_fast,
    verbose=True,
    max_iter=4,
    allow_delegation=False
)

content_strategist = Agent(
    role='Content Strategist',
    goal='Create a precise content brief: target keywords with volumes, a differentiated angle, reader persona, and structured outline.',
    backstory='Expert in SEO content strategy for e-commerce and tech. You find the intersection of high search intent and underserved angles.',
    tools=[keyword_data_tool, existing_posts_tool],
    llm=llm_fast,
    verbose=True,
    max_iter=3,
    allow_delegation=False
)

content_writer = Agent(
    role='Senior SEO Content Writer — Smart Home & Consumer Tech',
    goal='Write authoritative blog posts that rank on Google and convert readers into buyers. Technical accuracy. No fluff. Concrete specifics.',
    backstory='8 years writing in the smart home space. Your articles regularly rank top 3. Readers are consumers making purchase decisions — they want specs, prices, and honest trade-offs.',
    tools=[],
    llm=llm_quality,  # Best model for writing
    verbose=True,
    max_iter=3,
    allow_delegation=False
)

seo_specialist = Agent(
    role='Technical SEO Specialist',
    goal='Review content for SEO quality. Output revised title, meta description, and numbered specific improvements.',
    backstory='Worked on 200+ content sites. You think in search intent and E-E-A-T. You give specific numbered recommendations — never generic advice.',
    tools=[keyword_data_tool],
    llm=llm_fast,
    verbose=True,
    max_iter=3,
    allow_delegation=False
)

social_specialist = Agent(
    role='Social Media Content Specialist',
    goal='Adapt blog content into platform-native posts for Twitter/X, LinkedIn, and Reddit. Each platform has different norms — write natively for each.',
    backstory='You understand the culture of each platform. Twitter/X: punchy, thread-friendly. LinkedIn: professional insight. Reddit: genuinely helpful, never salesy.',
    tools=[],
    llm=llm_fast,
    verbose=True,
    max_iter=2,
    allow_delegation=False
)

print('✅ All 5 agents defined')

In [ ]:
# ── Tasks ────────────────────────────────────────────────────────────────

research_task = Task(
    description="""Research the following niche for Knovik's content team.
Niche: {niche} | Target site: {target_site}

Find: market size + growth rate, top 5 competitors with key differentiators,
main consumer problems solved, recent trends (last 6 months),
SEO keyword data for 3-5 primary keywords, content gaps competitors haven't covered.""",
    expected_output="""Structured research brief:
- Market overview (2-3 sentences)
- Competitor table: [name, price range, key feature, weakness]
- Consumer pain points: [problem, severity, solution]
- Keyword opportunities: [keyword, volume, difficulty, recommendation]
- Recommended content angle""",
    agent=market_researcher
)

strategy_task = Task(
    description="""Using the research, create a detailed content brief for the writer.
Niche: {niche} | Target site: {target_site} | Audience: {target_audience}

Include: recommended title, primary + 3 secondary keywords, reader persona,
differentiated angle, full H2/H3 outline, internal links, tone guide, word count target.""",
    expected_output='Complete content brief the writer can execute without clarification. Must include at least one internal linking opportunity.',
    agent=content_strategist,
    context=[research_task]
)

writing_task = Task(
    description="""Write a complete, publish-ready blog post following the content brief.
Requirements: follow the outline exactly, natural keyword use (no stuffing),
concrete product names and prices from research, compelling 2-sentence hook intro,
comparison table, CTA relevant to {target_site}, Markdown format.""",
    expected_output='Complete Markdown blog post: H1 title, all sections as H2/H3, 1200-1800 words, at least 1 comparison table, internal links, CTA.',
    agent=content_writer,
    context=[research_task, strategy_task]
)

seo_task = Task(
    description='Review the blog post for SEO quality. Check: title tag, meta description, keyword placement, header structure, missing keywords, internal links, E-E-A-T signals.',
    expected_output='SEO review: (1) optimized title tag, (2) meta description 150-160 chars, (3) numbered specific improvements, (4) SEO score 0-100 with justification, (5) revised post with all changes applied.',
    agent=seo_specialist,
    context=[writing_task]
)

social_task = Task(
    description='Create platform-native social posts from the blog content. Twitter/X thread (5-7 tweets), LinkedIn post (150-200 words), Reddit post for r/homekit or r/smarthome. Link to: {target_site}',
    expected_output='Three ready-to-post pieces formatted clearly: ## Twitter/X Thread, ## LinkedIn Post, ## Reddit Post (title + body)',
    agent=social_specialist,
    context=[writing_task, seo_task]
)

print('✅ All 5 tasks defined')

In [ ]:
# ── Assemble and Run ──────────────────────────────────────────────────────

content_crew = Crew(
    agents=[market_researcher, content_strategist, content_writer, seo_specialist, social_specialist],
    tasks=[research_task, strategy_task, writing_task, seo_task, social_task],
    process=Process.sequential,
    verbose=True,
    memory=False  # Set True to enable cross-task memory (requires embeddings)
)

console.print(Panel(
    '[bold cyan]Knovik Marketing Platform Content Crew[/]\n5 agents · 5 tasks · Sequential process',
    title='🚀 Starting'
))

result = content_crew.kickoff(inputs={
    'niche': 'Zigbee smart home hubs compatible with Apple HomeKit',
    'target_site': 'athbridge.com',
    'target_audience': 'Apple HomeKit users who want to add Zigbee devices to their smart home'
})

print('\n✅ Crew completed')

In [ ]:
# ── Display Final Output ──────────────────────────────────────────────────
console.print('\n' + '='*80)
console.print('[bold green]FINAL OUTPUT[/]')
console.print('='*80 + '\n')
console.print(Markdown(result.raw))

In [ ]:
# ── Per-task outputs ──────────────────────────────────────────────────────
task_names = ['Market Research', 'Content Strategy', 'Blog Post', 'SEO Review', 'Social Media']
for i, task_output in enumerate(result.tasks_output):
    preview = task_output.raw[:400] + '...' if len(task_output.raw) > 400 else task_output.raw
    console.print(Panel(preview, title=f'Task {i+1}: {task_names[i]}', border_style='dim'))

---

## Exercises

1. **Change the niche**: Try `"Matter protocol hubs"` or `"Thread border routers for HomeKit"`
2. **Change the target site**: Use `addtohomekit.com` — notice how the CTA adapts
3. **Add an agent**: Insert a `fact_checker` agent after the writer that verifies all product specs
4. **Switch to hierarchical**: Add a `Content Director` manager and change to `Process.hierarchical`
5. **Connect to Ghost CMS**: Wire up `save_ghost_draft` from the tools chapter as a final step

---

## Production notes for Marketing Platform

1. Replace `keyword_data_tool` simulation with a real DataForSEO or Ahrefs API call
2. Replace `existing_posts_tool` simulation with a live Strapi API query
3. Add `save_ghost_draft` as the final task so output goes directly to CMS
4. Only the writer needs `claude-3-5-sonnet` — use `claude-3-5-haiku` for all other agents to cut costs by ~50%
5. Set `LANGCHAIN_TRACING_V2=true` before running to get full LangSmith traces
6. Wrap `crew.kickoff()` in a task queue (Celery) for concurrent request handling
